*AI-generated draft (Claude, Anthropic) — for review. The labeling UI is version-controlled; the corrected boxes you draw are your own ground truth.*

<span style="font-family: 'Courier New', monospace;">

# 31 · Box-labeler — correct pre-labels for the v2 retrain

Each frame opens with mushroom.pt's **pre-labels** drawn as orange boxes. Because the model has no false positives but misses ~⅓ of worms, your job is mostly to **add the missed worms**.

**Kernel:** `joseph-scaleworm-thesis` (needs `ipympl`). Set **`UNIT`** in the setup cell to `"2022"` or `"2021"`.

**How to use**
1. **Drag** a rectangle around a worm the model missed → adds a box.
2. **Right-click** inside a box to delete it (e.g. a rare wrong one, or your own mistake).
3. **Undo box** removes the last one; **Clear** removes all.
4. **Save & Next ▶** writes the corrected YOLO label to `labels/train/` and copies the image to `images/train/`, then loads the next frame. **Skip frame** advances without saving (use for an unusable frame).
5. Zoom/pan with the toolbar, but **deselect zoom before drawing** boxes.

Resumable; opens on the first un-corrected frame. When done, carve ~15% into `images/val`+`labels/val`, then run `python scripts/train_v2.py`.
*(If box-drawing is finicky in your browser, Label Studio is the documented fallback — see `datasets/scaleworm_v2/README.md`.)*
</span>

In [ ]:
%matplotlib widget
import shutil
from pathlib import Path

import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from IPython.display import display

REPO = Path("/home/jovyan/scaleworm-student-lab")
DS = REPO / "datasets/scaleworm_v2"
UNIT = "2022"  # which pre-label batch to correct: "2022" or "2021"

PRE = DS / "prelabels" / UNIT
IMG_OUT = DS / "images/train"
LBL_OUT = DS / "labels/train"
IMG_OUT.mkdir(parents=True, exist_ok=True)
LBL_OUT.mkdir(parents=True, exist_ok=True)

stems = sorted(p.stem for p in (PRE / "images").glob("*.png"))
done = sum(1 for s in stems if (LBL_OUT / f"{s}.txt").exists())
print(f"{len(stems)} {UNIT} frames to correct  ({done} already corrected).")
print("Drag to ADD a box, right-click a box to DELETE, then Save & Next.")


def load_yolo(path, W, H):
    boxes = []
    if path.exists():
        for line in path.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cx, cy, w, h = (float(v) for v in p[1:])
                cx, cy, w, h = cx * W, cy * H, w * W, h * H
                boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return boxes


def save_yolo(path, boxes, W, H):
    lines = []
    for x1, y1, x2, y2 in boxes:
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        w, h = abs(x2 - x1) / W, abs(y2 - y1) / H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    path.write_text("\n".join(lines) + ("\n" if lines else ""))

In [ ]:
class BoxLabeler:
    """Correct mushroom.pt pre-labels: drag to add a box, right-click to delete."""

    def __init__(self, stems):
        self.stems = stems
        self.pos = next(
            (i for i, s in enumerate(stems) if not (LBL_OUT / f"{s}.txt").exists()), 0
        )
        self.boxes = []
        self.patches = []

        def mk(desc, style=""):
            return widgets.Button(description=desc, button_style=style,
                                  layout=widgets.Layout(width="auto"))

        self.b_undo = mk("Undo box")
        self.b_clear = mk("Clear")
        self.b_prev = mk("◀ Prev")
        self.b_save = mk("Save & Next ▶", "success")
        self.b_skip = mk("Skip frame", "warning")
        self.b_undo.on_click(lambda _: self._undo())
        self.b_clear.on_click(lambda _: self._clear())
        self.b_prev.on_click(lambda _: self._prev())
        self.b_save.on_click(lambda _: self._save())
        self.b_skip.on_click(lambda _: self._advance())
        self.status = widgets.HTML()
        self.msg = widgets.Output()

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 7))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_position = "right"
        self.selector = RectangleSelector(
            self.ax, self._on_select, useblit=True, button=[1],
            minspanx=3, minspany=3, spancoords="pixels", interactive=False,
            props=dict(facecolor="none", edgecolor="#00A000", linewidth=1.5),
        )
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)

        controls = widgets.HBox(
            [self.b_undo, self.b_clear, self.b_prev, self.b_save, self.b_skip]
        )
        self.box = widgets.VBox([controls, self.status, self.fig.canvas, self.msg])
        self._load()

    def _stem(self):
        return self.stems[self.pos]

    def _load(self):
        stem = self._stem()
        img = mpimg.imread(PRE / "images" / f"{stem}.png")
        self.H, self.W = img.shape[:2]
        self.ax.clear()
        self.ax.imshow(img)
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        src = LBL_OUT / f"{stem}.txt"
        if not src.exists():
            src = PRE / "labels" / f"{stem}.txt"
        self.boxes = load_yolo(src, self.W, self.H)
        self._draw()
        self._status()

    def _draw(self):
        for p in self.patches:
            try:
                p.remove()
            except ValueError:
                pass
        self.patches = []
        for x1, y1, x2, y2 in self.boxes:
            r = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                   edgecolor="#D55E00", linewidth=1.6)
            self.ax.add_patch(r)
            self.patches.append(r)
        self.ax.set_title(f"{self._stem()}   —   {len(self.boxes)} boxes", fontsize=10)
        self.fig.canvas.draw_idle()

    def _on_select(self, eclick, erelease):
        x1, y1, x2, y2 = eclick.xdata, eclick.ydata, erelease.xdata, erelease.ydata
        if None in (x1, y1, x2, y2):
            return
        self.boxes.append([min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)])
        self._draw()

    def _on_click(self, event):
        if event.button == 3 and event.inaxes == self.ax and event.xdata is not None:
            hit = [i for i, (x1, y1, x2, y2) in enumerate(self.boxes)
                   if x1 <= event.xdata <= x2 and y1 <= event.ydata <= y2]
            if hit:
                self.boxes.pop(hit[-1])
                self._draw()

    def _undo(self):
        if self.boxes:
            self.boxes.pop()
            self._draw()

    def _clear(self):
        self.boxes = []
        self._draw()

    def _status(self):
        self.status.value = (
            f"<b>Frame {self.pos + 1}/{len(self.stems)}</b> &nbsp; {self._stem()}"
        )

    def _save(self):
        stem = self._stem()
        save_yolo(LBL_OUT / f"{stem}.txt", self.boxes, self.W, self.H)
        shutil.copy(PRE / "images" / f"{stem}.png", IMG_OUT / f"{stem}.png")
        self._advance()

    def _advance(self):
        self.msg.clear_output()
        if self.pos < len(self.stems) - 1:
            self.pos += 1
            self._load()
        else:
            with self.msg:
                print("✅ All frames reviewed. Carve ~15% into images/val+labels/val, "
                      "then run scripts/train_v2.py")

    def _prev(self):
        if self.pos > 0:
            self.pos -= 1
            self._load()


app = BoxLabeler(stems)
display(app.box)